# RF・Logistic・Shuffleの健全性診断

歴史的な実験コードです。現在の実行入口は `../09_confidence_nested.ipynb`。
元Notebookのセル番号は0始まりです。コードの個人フォルダ名は置換しています。独立実行は保証しません。
保存出力は `../../results/imported_20260907/`、監査は `../../docs/CONFIDENCE_AUDIT.md` を参照してください。


## 元のセル index 32


In [ ]:
# ============================================================
# 15分足10年 ML健全性テスト
#
# RandomForest vs Logistic vs Shuffle
#
# 目的
# ------------------------------------------------------------
# 「5分足ではノイズを過学習していたのでは？」
# という仮説を検証するため、
# まず15分足10年データで
# 機械学習そのものに予測能力があるか調べる。
#
# 予測対象:
# 30分後にUSD/JPYが上か下か
#
# 重要:
# Testは学習に一切使用しない。
# ============================================================


from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score


# ============================================================
# 1. 設定
# ============================================================

CSV_PATH = (
    Path.cwd()
    / "dukascopy_usdjpy"
    / "usdjpy_15m_2016_2026.csv"
)

# 15分足 × 2本 = 30分後
HORIZON_BARS = 2

N_FOLDS = 5

RANDOM_STATE = 42

# 利益計算用。
# とりあえず往復コスト0.004%を仮置き。
# 後で実際の業者に合わせて変更する。
TRADING_COST = 0.00004


# ============================================================
# 2. CSV読み込み
# ============================================================

df = pd.read_csv(
    CSV_PATH,
    index_col=0,
    parse_dates=True
)

df.index = pd.to_datetime(
    df.index,
    utc=True
)

df = df.sort_index()


# 列名統一
df.columns = [
    c.lower()
    for c in df.columns
]


print("読み込み完了")
print("総行数:", len(df))
print("期間:", df.index.min(), "→", df.index.max())
print("columns:", df.columns.tolist())


# ============================================================
# 3. RSI
# ============================================================

def calculate_rsi(
    close,
    period=14
):

    delta = close.diff()

    gain = (
        delta.clip(
            lower=0
        )
    )

    loss = (
        -delta.clip(
            upper=0
        )
    )

    avg_gain = (
        gain
        .rolling(period)
        .mean()
    )

    avg_loss = (
        loss
        .rolling(period)
        .mean()
    )

    rs = (
        avg_gain
        /
        avg_loss.replace(
            0,
            np.nan
        )
    )

    return (
        100
        -
        100
        /
        (
            1 + rs
        )
    )


# ============================================================
# 4. 特徴量作成
#
# 全て現在または過去だけから計算
# ============================================================

def make_features(
    data
):

    x = data.copy()

    # --------------------------------------------------------
    # Return
    # --------------------------------------------------------

    x["return_1"] = (
        x["close"]
        .pct_change(1)
    )

    x["return_2"] = (
        x["close"]
        .pct_change(2)
    )

    x["return_4"] = (
        x["close"]
        .pct_change(4)
    )

    x["return_8"] = (
        x["close"]
        .pct_change(8)
    )

    x["return_16"] = (
        x["close"]
        .pct_change(16)
    )

    # --------------------------------------------------------
    # Volatility
    # --------------------------------------------------------

    x["vol_4"] = (
        x["return_1"]
        .rolling(4)
        .std()
    )

    x["vol_8"] = (
        x["return_1"]
        .rolling(8)
        .std()
    )

    x["vol_16"] = (
        x["return_1"]
        .rolling(16)
        .std()
    )

    x["vol_32"] = (
        x["return_1"]
        .rolling(32)
        .std()
    )

    # --------------------------------------------------------
    # Moving Average
    # --------------------------------------------------------

    for period in [
        5,
        10,
        20,
        50,
        100
    ]:

        ma = (
            x["close"]
            .rolling(period)
            .mean()
        )

        x[
            f"ma{period}_distance"
        ] = (
            x["close"]
            /
            ma
            - 1
        )

        x[
            f"ma{period}_slope"
        ] = (
            ma.pct_change()
        )

    # --------------------------------------------------------
    # Candle形状
    # --------------------------------------------------------

    candle_range = (
        x["high"]
        -
        x["low"]
    ).replace(
        0,
        np.nan
    )

    x["body"] = (
        x["close"]
        -
        x["open"]
    ) / candle_range

    x["upper_wick"] = (
        x["high"]
        -
        x[
            [
                "open",
                "close"
            ]
        ].max(
            axis=1
        )
    ) / candle_range

    x["lower_wick"] = (
        x[
            [
                "open",
                "close"
            ]
        ].min(
            axis=1
        )
        -
        x["low"]
    ) / candle_range

    x["range_pct"] = (
        x["high"]
        -
        x["low"]
    ) / x["close"]

    # --------------------------------------------------------
    # RSI
    # --------------------------------------------------------

    x["rsi14"] = (
        calculate_rsi(
            x["close"],
            14
        )
        / 100
    )

    # --------------------------------------------------------
    # ATR風
    # --------------------------------------------------------

    previous_close = (
        x["close"]
        .shift(1)
    )

    true_range = pd.concat(
        [
            x["high"]
            -
            x["low"],

            (
                x["high"]
                -
                previous_close
            ).abs(),

            (
                x["low"]
                -
                previous_close
            ).abs(),
        ],
        axis=1
    ).max(
        axis=1
    )

    x["atr14"] = (
        true_range
        .rolling(14)
        .mean()
        /
        x["close"]
    )

    # --------------------------------------------------------
    # 高値・安値までの距離
    # --------------------------------------------------------

    high_16 = (
        x["high"]
        .rolling(16)
        .max()
    )

    low_16 = (
        x["low"]
        .rolling(16)
        .min()
    )

    x["distance_high_16"] = (
        high_16
        -
        x["close"]
    ) / x["close"]

    x["distance_low_16"] = (
        x["close"]
        -
        low_16
    ) / x["close"]

    # --------------------------------------------------------
    # 時刻
    # --------------------------------------------------------

    hour = (
        x.index.hour
        +
        x.index.minute
        / 60
    )

    x["hour_sin"] = np.sin(
        2
        *
        np.pi
        *
        hour
        /
        24
    )

    x["hour_cos"] = np.cos(
        2
        *
        np.pi
        *
        hour
        /
        24
    )

    x["weekday"] = (
        x.index.dayofweek
        / 4
    )

    return x


data = make_features(
    df
)


# ============================================================
# 5. 30分後の教師ラベル
# ============================================================

data["future_return"] = (
    data["close"]
    .shift(
        -HORIZON_BARS
    )
    /
    data["close"]
    - 1
)

data["target"] = (
    data["future_return"]
    > 0
).astype(int)


# ============================================================
# 6. 使用特徴量
# ============================================================

FEATURES = [

    "return_1",
    "return_2",
    "return_4",
    "return_8",
    "return_16",

    "vol_4",
    "vol_8",
    "vol_16",
    "vol_32",

    "ma5_distance",
    "ma5_slope",

    "ma10_distance",
    "ma10_slope",

    "ma20_distance",
    "ma20_slope",

    "ma50_distance",
    "ma50_slope",

    "ma100_distance",
    "ma100_slope",

    "body",
    "upper_wick",
    "lower_wick",
    "range_pct",

    "rsi14",
    "atr14",

    "distance_high_16",
    "distance_low_16",

    "hour_sin",
    "hour_cos",
    "weekday",
]


data = (
    data
    .replace(
        [
            np.inf,
            -np.inf
        ],
        np.nan
    )
    .dropna(
        subset=
            FEATURES
            +
            [
                "future_return",
                "target"
            ]
    )
    .copy()
)


print()
print("ML使用可能データ:", len(data))
print("特徴量数:", len(FEATURES))
print("上昇割合:", data["target"].mean())


# ============================================================
# 7. Walk-Forward Fold
#
# 前半から順に学習を増やしていく
# ============================================================

block = (
    len(data)
    //
    (
        N_FOLDS
        + 1
    )
)


results = []

trade_results = []


for fold in range(
    1,
    N_FOLDS + 1
):

    print()
    print(
        "===================================="
    )

    print(
        f"Fold {fold}"
    )

    print(
        "===================================="
    )

    train_end = (
        block
        *
        fold
    )

    # 未来ラベル跨ぎ防止
    test_start = (
        train_end
        +
        HORIZON_BARS
    )

    test_end = min(
        test_start
        +
        block,
        len(data)
    )

    train = (
        data.iloc[
            :train_end
        ]
        .copy()
    )

    test = (
        data.iloc[
            test_start:test_end
        ]
        .copy()
    )

    if (
        len(train) < 1000
        or
        len(test) == 0
    ):
        continue

    X_train = (
        train[
            FEATURES
        ]
    )

    y_train = (
        train[
            "target"
        ]
    )

    X_test = (
        test[
            FEATURES
        ]
    )

    y_test = (
        test[
            "target"
        ]
    )

    # ========================================================
    # 8. Random Forest
    # ========================================================

    rf = RandomForestClassifier(

        n_estimators=
            300,

        max_depth=
            8,

        min_samples_leaf=
            30,

        max_features=
            "sqrt",

        class_weight=
            "balanced",

        random_state=
            RANDOM_STATE,

        n_jobs=
            -1,
    )

    rf.fit(
        X_train,
        y_train
    )


    rf_train_prob = (
        rf.predict_proba(
            X_train
        )[:, 1]
    )

    rf_test_prob = (
        rf.predict_proba(
            X_test
        )[:, 1]
    )


    rf_train_auc = (
        roc_auc_score(
            y_train,
            rf_train_prob
        )
    )

    rf_test_auc = (
        roc_auc_score(
            y_test,
            rf_test_prob
        )
    )


    # ========================================================
    # 9. Logistic Regression
    # ========================================================

    logistic = Pipeline(
        [
            (
                "scaler",
                StandardScaler()
            ),

            (
                "model",
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    random_state=
                        RANDOM_STATE
                )
            )
        ]
    )


    logistic.fit(
        X_train,
        y_train
    )


    log_train_prob = (
        logistic.predict_proba(
            X_train
        )[:, 1]
    )

    log_test_prob = (
        logistic.predict_proba(
            X_test
        )[:, 1]
    )


    log_train_auc = (
        roc_auc_score(
            y_train,
            log_train_prob
        )
    )

    log_test_auc = (
        roc_auc_score(
            y_test,
            log_test_prob
        )
    )


    # ========================================================
    # 10. Shuffle RF
    #
    # ラベルを完全にランダム化
    # ========================================================

    rng = np.random.default_rng(
        RANDOM_STATE
        +
        fold
    )

    shuffled_y = (
        rng.permutation(
            y_train.values
        )
    )


    shuffle_rf = RandomForestClassifier(

        n_estimators=
            200,

        max_depth=
            8,

        min_samples_leaf=
            30,

        max_features=
            "sqrt",

        class_weight=
            "balanced",

        random_state=
            RANDOM_STATE
            +
            fold,

        n_jobs=
            -1,
    )


    shuffle_rf.fit(
        X_train,
        shuffled_y
    )


    shuffle_test_prob = (
        shuffle_rf.predict_proba(
            X_test
        )[:, 1]
    )


    shuffle_auc = (
        roc_auc_score(
            y_test,
            shuffle_test_prob
        )
    )


    # ========================================================
    # 11. Accuracy
    # ========================================================

    rf_accuracy = (
        accuracy_score(
            y_test,
            (
                rf_test_prob
                >=
                0.5
            ).astype(int)
        )
    )


    log_accuracy = (
        accuracy_score(
            y_test,
            (
                log_test_prob
                >=
                0.5
            ).astype(int)
        )
    )


    # ========================================================
    # 12. 単純利益テスト
    #
    # TP/SLなし
    # Confidence最適化なし
    #
    # RFがUPならBUY
    # DOWNならSELL
    # ========================================================

    directions = np.where(
        rf_test_prob
        >= 0.5,
        1,
        -1
    )


    returns = (
        test[
            "future_return"
        ].values
        *
        directions
        -
        TRADING_COST
    )


    # 30分ポジション重複を避けるため
    # 2本ごとに使用
    returns = (
        returns[
            ::HORIZON_BARS
        ]
    )


    gains = (
        returns[
            returns > 0
        ].sum()
    )

    losses = (
        -returns[
            returns < 0
        ].sum()
    )


    if losses > 0:

        pf = (
            gains
            /
            losses
        )

    else:

        pf = np.inf


    avg_return = (
        returns.mean()
    )


    win_rate = (
        returns > 0
    ).mean()


    # ========================================================
    # 13. 保存
    # ========================================================

    row = {

        "fold":
            fold,

        "train_size":
            len(train),

        "test_size":
            len(test),

        "rf_train_auc":
            rf_train_auc,

        "rf_test_auc":
            rf_test_auc,

        "rf_gap":
            rf_train_auc
            -
            rf_test_auc,

        "log_train_auc":
            log_train_auc,

        "log_test_auc":
            log_test_auc,

        "log_gap":
            log_train_auc
            -
            log_test_auc,

        "shuffle_auc":
            shuffle_auc,

        "rf_accuracy":
            rf_accuracy,

        "log_accuracy":
            log_accuracy,

        "trades":
            len(returns),

        "win_rate":
            win_rate,

        "avg_return":
            avg_return,

        "profit_factor":
            pf,
    }


    results.append(
        row
    )


    print(
        "RF Train AUC:",
        round(
            rf_train_auc,
            4
        )
    )

    print(
        "RF Test AUC :",
        round(
            rf_test_auc,
            4
        )
    )

    print(
        "RF Gap      :",
        round(
            rf_train_auc
            -
            rf_test_auc,
            4
        )
    )


    print()

    print(
        "Log Test AUC:",
        round(
            log_test_auc,
            4
        )
    )

    print(
        "Shuffle AUC :",
        round(
            shuffle_auc,
            4
        )
    )

    print()

    print(
        "RF利益:"
    )

    print(
        "Trades:",
        len(returns)
    )

    print(
        "Win:",
        round(
            win_rate
            *
            100,
            2
        ),
        "%"
    )

    print(
        "Average:",
        round(
            avg_return
            *
            100,
            5
        ),
        "%"
    )

    print(
        "PF:",
        round(
            pf,
            3
        )
    )


# ============================================================
# 14. Fold結果
# ============================================================

results_df = pd.DataFrame(
    results
)


print()
print(
    "===================================="
)

print(
    "Fold別結果"
)

print(
    "===================================="
)


print(
    results_df.to_string(
        index=False
    )
)


# ============================================================
# 15. 平均
# ============================================================

print()
print(
    "===================================="
)

print(
    "平均結果"
)

print(
    "===================================="
)


summary_columns = [

    "rf_train_auc",
    "rf_test_auc",
    "rf_gap",

    "log_train_auc",
    "log_test_auc",
    "log_gap",

    "shuffle_auc",

    "rf_accuracy",
    "log_accuracy",

    "win_rate",
    "avg_return",
    "profit_factor",
]


summary = (
    results_df[
        summary_columns
    ]
    .mean()
)


print(
    summary
)


# ============================================================
# 16. 過学習判定
# ============================================================

mean_train = (
    results_df[
        "rf_train_auc"
    ].mean()
)

mean_test = (
    results_df[
        "rf_test_auc"
    ].mean()
)

mean_gap = (
    results_df[
        "rf_gap"
    ].mean()
)

mean_shuffle = (
    results_df[
        "shuffle_auc"
    ].mean()
)


print()
print(
    "===================================="
)

print(
    "過学習診断"
)

print(
    "===================================="
)


print(
    "RF Train AUC:",
    mean_train
)

print(
    "RF Test AUC:",
    mean_test
)

print(
    "Train-Test Gap:",
    mean_gap
)

print(
    "Shuffle AUC:",
    mean_shuffle
)


print()


if (
    mean_test
    < 0.52
):

    print(
        "判定:"
    )

    print(
        "現在の特徴量では未来方向の予測能力がかなり弱いです。"
    )

elif (
    mean_test
    < 0.55
):

    print(
        "判定:"
    )

    print(
        "弱いシグナルはありますが、まだ実用には不足しています。"
    )

elif (
    mean_test
    < 0.60
):

    print(
        "判定:"
    )

    print(
        "予測可能性が確認できます。研究継続価値があります。"
    )

else:

    print(
        "判定:"
    )

    print(
        "かなり強い予測能力が出ています。"
    )


if (
    mean_gap
    > 0.10
):

    print()

    print(
        "Train-Test差が大きいため、"
    )

    print(
        "RandomForestがノイズを過学習している可能性が高いです。"
    )


# ============================================================
# 17. Fold安定性
# ============================================================

positive_auc_folds = (
    results_df[
        "rf_test_auc"
    ]
    >
    0.5
).sum()


auc_52_folds = (
    results_df[
        "rf_test_auc"
    ]
    >
    0.52
).sum()


auc_55_folds = (
    results_df[
        "rf_test_auc"
    ]
    >
    0.55
).sum()


print()
print(
    "===================================="
)

print(
    "Fold安定性"
)

print(
    "===================================="
)


print(
    "AUC > 0.50:",
    positive_auc_folds,
    "/",
    len(
        results_df
    )
)

print(
    "AUC > 0.52:",
    auc_52_folds,
    "/",
    len(
        results_df
    )
)

print(
    "AUC > 0.55:",
    auc_55_folds,
    "/",
    len(
        results_df
    )
)


# ============================================================
# 18. AUCグラフ
# ============================================================

plt.figure(
    figsize=(
        9,
        5
    )
)


plt.plot(
    results_df[
        "fold"
    ],
    results_df[
        "rf_train_auc"
    ],
    marker="o",
    label="RF Train"
)


plt.plot(
    results_df[
        "fold"
    ],
    results_df[
        "rf_test_auc"
    ],
    marker="o",
    label="RF Test"
)


plt.plot(
    results_df[
        "fold"
    ],
    results_df[
        "log_test_auc"
    ],
    marker="o",
    label="Logistic Test"
)


plt.plot(
    results_df[
        "fold"
    ],
    results_df[
        "shuffle_auc"
    ],
    marker="o",
    label="Shuffle"
)


plt.axhline(
    0.5,
    linewidth=1
)


plt.xlabel(
    "Fold"
)

plt.ylabel(
    "ROC-AUC"
)

plt.title(
    "15m / 10-year Model Health Check"
)

plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# 19. 保存
# ============================================================

OUTPUT_DIR = (
    Path.cwd()
    /
    "15m_model_health_check"
)

OUTPUT_DIR.mkdir(
    exist_ok=True
)


results_df.to_csv(
    OUTPUT_DIR
    /
    "fold_results.csv",
    index=False
)


print()
print(
    "保存先:"
)

print(
    OUTPUT_DIR.resolve()
)